# T5 Fine-Tuning for Medical Jargon Simplification

**Pipeline position:**
```
User Input
→ NLP Term Extraction        (jargonbackend.ipynb)
→ RAG Retrieval (FAISS)      (jargonbackend.ipynb)
→ T5 Simplification          ← THIS NOTEBOOK
→ Readability Score          (main.py)
→ Confidence Score           (main.py)
→ Human Validation           (future)
```

**Input:**  `datasets/simplification.json`
**Output:** `models/t5-medical-finetuned/` (loaded by main.py at runtime)

## Step 1 — Install Dependencies

## Step 1 — Install Dependencies

In [2]:
!pip install transformers datasets evaluate rouge_score accelerate sentencepiece wandb huggingface_hub scikit-learn -q

## Step 2 — Imports

In [3]:
import os
import json
import torch
import numpy as np
import pandas as pd
from getpass import getpass

from datasets import Dataset, DatasetDict
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
import evaluate

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('No GPU — training will be slow. Use Google Colab with T4 GPU.')

/Users/sheetalthapa/Downloads/Minor-Project-/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
No GPU — training will be slow. Use Google Colab with T4 GPU.


## Step 3 — Login (optional)

In [4]:
# Only needed if uploading to HuggingFace Hub — skip if saving locally
PUSH_TO_HUB = False
USE_WANDB   = False

if PUSH_TO_HUB:
    from huggingface_hub import login
    hf_token = getpass("Enter HuggingFace token: ")
    login(token=hf_token)
    print("HuggingFace login OK")

if USE_WANDB:
    import wandb
    wandb_key = getpass("Enter WandB key: ")
    wandb.login(key=wandb_key)
    print("WandB login OK")
else:
    os.environ["WANDB_DISABLED"] = "true"
    print("WandB disabled")

WandB disabled


## Step 4 — Load & Inspect Dataset

In [5]:
DATASET_PATH  = "../datasets/simplification.json"
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

print(f"Total records: {len(raw)}")
print(f"First record:")
print(json.dumps(raw[0], indent=2))

Total records: 4459
First record:
{
  "doi": "10.1002/14651858.CD001836.pub4",
  "abstract": "Two trials met the inclusion criteria. One compared 2% ketanserin ointment in polyethylene glycol (PEG) with PEG alone, used twice a day by 40 participants with arterial leg ulcers, for eight weeks or until healing, whichever was sooner. One compared topical application of blood-derived concentrated growth factor (CGF) with standard dressing (polyurethane film or foam); both applied weekly for six weeks by 61 participants with non-healing ulcers (venous, diabetic arterial, neuropathic, traumatic, or vasculitic). Both trials were small, reported results inadequately, and were of low methodological quality. Short follow-up times (six and eight weeks) meant it would be difficult to capture sufficient healing events to allow us to make comparisons between treatments. One trial demonstrated accelerated wound healing in the ketanserin group compared with the control group. In the trial that compared

In [6]:
# Check keys in your JSON
if isinstance(raw, list):
    print("Keys found:", list(raw[0].keys()))
elif isinstance(raw, dict):
    print("Top-level keys:", list(raw.keys()))
    for k in raw:
        print(f"  {k}: {len(raw[k])} records")

Keys found: ['doi', 'abstract', 'pls']


In [7]:
print(json.dumps(raw[0], indent=2))

{
  "doi": "10.1002/14651858.CD001836.pub4",
  "abstract": "Two trials met the inclusion criteria. One compared 2% ketanserin ointment in polyethylene glycol (PEG) with PEG alone, used twice a day by 40 participants with arterial leg ulcers, for eight weeks or until healing, whichever was sooner. One compared topical application of blood-derived concentrated growth factor (CGF) with standard dressing (polyurethane film or foam); both applied weekly for six weeks by 61 participants with non-healing ulcers (venous, diabetic arterial, neuropathic, traumatic, or vasculitic). Both trials were small, reported results inadequately, and were of low methodological quality. Short follow-up times (six and eight weeks) meant it would be difficult to capture sufficient healing events to allow us to make comparisons between treatments. One trial demonstrated accelerated wound healing in the ketanserin group compared with the control group. In the trial that compared CGF with standard dressings, the 

In [8]:
# ---- CHANGE THESE to match your actual JSON keys ----
INPUT_KEY  = "abstract"  # full medical jargon text
TARGET_KEY = "pls"       # plain language simplified version
# -----------------------------------------------------

# Handle both flat list and split dict formats
if isinstance(raw, list):
    records = raw
else:
    records = []
    for v in raw.values():
        records.extend(v)

# Build normalised list
data = []
for r in records:
    inp = str(r.get(INPUT_KEY, "") or "").strip()
    tgt = str(r.get(TARGET_KEY, "") or "").strip()
    if inp and tgt:
        data.append({"input_text": inp, "target_text": tgt})

df = pd.DataFrame(data)
print(f"Clean records: {len(df)}")
print(f"Input avg length:  {df.input_text.str.len().mean():.0f} chars")
print(f"Target avg length: {df.target_text.str.len().mean():.0f} chars")
df.head()

Clean records: 4459
Input avg length:  2289 chars
Target avg length: 1370 chars


,input_text,target_text
0,Two trials met the inclusion criteria. One com...,We found two small studies that presented data...
1,We identified one RCT that involved 40 partici...,The searches are up-to-date to 26 January 2016...
2,We included 13 studies with a total of 721 par...,The benefits of PN are uncertain as the eviden...
3,We included 25 RCTs involving 4788 participant...,The findings of this review were inconclusive ...
4,We conducted this review in accordance with th...,We searched published medical articles to find...


## Step 5 — Train / Validation Split

In [9]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)}  |  Validation: {len(val_df)}")

train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)

dataset = DatasetDict({"train": train_dataset, "validation": val_dataset})
print(dataset)

Train: 3790  |  Validation: 669
DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 3790
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 669
    })
})


## Step 6 — Load T5 Model & Tokeniser

In [10]:
# t5-small → fast, less accurate (good for testing on CPU)
# t5-base  → balanced (recommended for your project)
# t5-large → best quality, needs 8GB+ GPU RAM

MODEL_NAME = "t5-small"

print(f"Loading {MODEL_NAME}...")
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model     = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model     = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded — {total_params/1e6:.0f}M parameters")

Loading t5-small...


Loading weights: 100%|██████████| 131/131 [00:00<00:00, 2725.87it/s, Materializing param=shared.weight]                                                      


Model loaded — 61M parameters


## Step 7 — Tokenise Dataset

In [11]:
PREFIX     = "simplify medical text: "
MAX_INPUT  = 512   # increased — abstracts are long paragraphs
MAX_TARGET = 256   # increased — pls summaries are also longer than 128

def tokenise(batch):
    inputs = [PREFIX + t for t in batch["input_text"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        batch["target_text"],
        max_length=MAX_TARGET,
        truncation=True,
        padding="max_length",
    )

    # Replace padding id with -100 so it is ignored in loss
    labels_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels_ids
    return model_inputs

print("Tokenising...")
tokenised = dataset.map(
    tokenise,
    batched=True,
    remove_columns=["input_text", "target_text"],
)
print("Done")
print(tokenised)

Tokenising...


Map: 100%|██████████| 669/669 [00:00<00:00, 2391.54 examples/s]

Done
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3790
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 669
    })
})


## Step 8 — ROUGE Evaluation Metric

In [12]:
# ROUGE measures overlap between generated and reference text
# ROUGE-1: word overlap | ROUGE-2: phrase overlap | ROUGE-L: sequence match
# Higher = better simplification quality. Target: ROUGE-L > 0.40

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )
    return {
        "rouge1": round(result["rouge1"], 4),
        "rouge2": round(result["rouge2"], 4),
        "rougeL": round(result["rougeL"], 4),
    }

print("ROUGE metric ready")

ROUGE metric ready


## Step 9 — Training Arguments

In [13]:
OUTPUT_DIR  = "../models/t5-medical-finetuned"  # save outside notebook folder
LOGGING_DIR = "../models/t5-logs"
os.makedirs(OUTPUT_DIR,  exist_ok=True)
os.makedirs(LOGGING_DIR, exist_ok=True)

# You are on CPU — keeping it small so it actually finishes
BATCH_SIZE    = 2
GRAD_ACCUM    = 4    # effective batch = 8
NUM_EPOCHS    = 3    # 5 epochs on CPU would take many hours
LEARNING_RATE = 3e-4

training_args = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    warmup_steps                = 50,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",   # renamed from evaluation_strategy
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "rougeL",
    greater_is_better           = True,
    predict_with_generate       = True,
    generation_max_length       = MAX_TARGET,
    logging_dir                 = LOGGING_DIR,
    logging_steps               = 20,
    report_to                   = "none",
    save_total_limit            = 2,
    fp16                        = False,
)
print(f"Epochs:          {NUM_EPOCHS}")
print(f"Batch size:      {BATCH_SIZE}")
print(f"Grad accum:      {GRAD_ACCUM}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Output dir:      {OUTPUT_DIR}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epochs:          3
Batch size:      2
Grad accum:      4
Effective batch: 8
Output dir:      ../models/t5-medical-finetuned


## Step 10 — Train

In [14]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=None,
)

trainer = Seq2SeqTrainer(
    model             = model,
    args              = training_args,
    train_dataset     = tokenised["train"],
    eval_dataset      = tokenised["validation"],
    processing_class  = tokenizer,        # renamed from tokenizer
    data_collator     = data_collator,
    compute_metrics   = compute_metrics,
    callbacks         = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training...")
print("Watch ROUGE-L score improve each epoch.")

train_result = trainer.train()

print("Training complete!")
print(f"Final train loss: {train_result.training_loss:.4f}")

Starting training...
Watch ROUGE-L score improve each epoch.


/Users/sheetalthapa/Downloads/Minor-Project-/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


OverflowError: out of range integral type conversion attempted

## Step 11 — Evaluate

In [ ]:
metrics = trainer.evaluate()

print("── Evaluation Results ─────────────────")
print(f"  ROUGE-1 : {metrics.get('eval_rouge1', 0):.4f}  (word overlap)")
print(f"  ROUGE-2 : {metrics.get('eval_rouge2', 0):.4f}  (phrase overlap)")
print(f"  ROUGE-L : {metrics.get('eval_rougeL', 0):.4f}  (sequence match)")
print(f"  Val Loss: {metrics.get('eval_loss',   0):.4f}")
print("───────────────────────────────────────")
print("Target: ROUGE-L > 0.40 = good simplification")

## Step 12 — Save Model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(f"{OUTPUT_DIR}/eval_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Model saved to: {OUTPUT_DIR}/")
print("Files:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fsize = os.path.getsize(f"{OUTPUT_DIR}/{fname}") / 1024**2
    print(f"  {fname}  ({fsize:.1f} MB)")

## Step 13 — Test Inference

In [ ]:
print("Loading saved model for inference test...")
test_tok   = T5Tokenizer.from_pretrained(OUTPUT_DIR)
test_model = T5ForConditionalGeneration.from_pretrained(OUTPUT_DIR).to(device)
test_model.eval()

def t5_simplify(text, max_length=128):
    input_ids = test_tok(
        PREFIX + text,
        return_tensors="pt",
        max_length=MAX_INPUT,
        truncation=True,
    ).input_ids.to(device)

    with torch.no_grad():
        out = test_model.generate(
            input_ids,
            max_length=max_length,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
    return test_tok.decode(out[0], skip_special_tokens=True)

test_inputs = [
    "The patient presents with acute myocardial infarction.",
    "Hypertension is a chronic condition requiring antihypertensive therapy.",
    "The diagnosis indicates bilateral pneumonia with pleural effusion.",
]

print("── Inference Tests ────────────────────────────────")
for text in test_inputs:
    print(f"Input:  {text}")
    print(f"Output: {t5_simplify(text)}")
print("────────────────────────────────────────────────────")

## Step 14 — S_source Faithfulness Score

In [ ]:
import sqlite3

def compute_s_source(term, t5_output):
    """
    Compare T5 output vs original DB content for this term.
    Returns ROUGE-L as S_source used in:
    C_final = 0.4*S_retrieval + 0.4*S_source + 0.2*S_human
    """
    try:
        conn = sqlite3.connect("medical_jargon.db")
        row  = conn.execute(
            "SELECT content FROM medical_terms WHERE LOWER(term)=LOWER(?)",
            (term,)
        ).fetchone()
        conn.close()

        if not row or not row[0]:
            return 0.8   # default if no source to compare

        result = rouge.compute(
            predictions=[t5_output],
            references=[row[0]],
            use_stemmer=True,
        )
        return round(result["rougeL"], 4)
    except Exception as e:
        print(f"S_source error: {e}")
        return 0.8

# Test
term   = "hypertension"
t5_out = t5_simplify("The patient has hypertension requiring medication.")
s_src  = compute_s_source(term, t5_out)
print(f"Term:      {term}")
print(f"T5 output: {t5_out}")
print(f"S_source:  {s_src}  (faithfulness to DB source)")

## Step 15 — Code to Add to main.py After Training



Then in  route after FAISS retrieval:
